In [2]:
import pandas as pd
from gqlalchemy import Memgraph
from pathlib import Path

In [2]:
alzkb = Memgraph(host='alzkb.ai', port=7687)

In [3]:
def get_associated_entities(drug, gene_list, output_dir):
	query = alzkb.execute_and_fetch("MATCH (a:Gene) Return a.nodeID,a.geneSymbol,a.chromosome, a.commonName,a.typeOfGene, a.xrefEnsembl ")
	genes = pd.DataFrame(query)
	genes_sub = genes[genes['a.geneSymbol'].isin(gene_list)]

	drugs = alzkb.execute_and_fetch("MATCH (a:Drug) Return a.nodeID,a.xrefDrugbank, a.commonName ")
	drugs = pd.DataFrame(drugs)
	# print(drugs)
	drugs_sub =drugs[drugs['a.xrefDrugbank'].eq(drug)]
	print(drugs_sub)

	# find other drugs connected to the list of genes
	gene_drug =  alzkb.execute_and_fetch("MATCH (a:Gene)-[]-(b:Drug) Return a.nodeID,a.geneSymbol, a.commonName, b.commonName, b.xrefDrugbank")
	gene_drug= pd.DataFrame(gene_drug)
	gene_drug_sub =gene_drug[gene_drug['a.geneSymbol'].isin(gene_list)]
	gene_drug_sub.to_csv(output_dir / f'{drug}_other_drugs.csv', index=False)
	gene_drug_sub['b.commonName'].value_counts().to_csv(output_dir / f'{drug}_drugs_valueCounts.csv', index=True)

	# find associated pathways
	gene_path =  alzkb.execute_and_fetch("MATCH (a:Gene)-[]-(b:Pathway) Return a.nodeID,a.geneSymbol, a.commonName, b.nodeID, b.pathwayName")
	gene_path= pd.DataFrame(gene_path)
	gene_path_sub =gene_path[gene_path['a.geneSymbol'].isin(gene_list)]
	#print(gene_path_sub)
	gene_path_sub.to_csv(output_dir / f'{drug}_pathways.csv', index=False)
	gene_path_sub['b.pathwayName'].value_counts().to_csv(output_dir / f'{drug}_pathways_valueCounts.csv', index=True)

	# find associated body parts
	gene_bp =  alzkb.execute_and_fetch("MATCH (a:Gene)-[]-(b:BodyPart) Return a.nodeID,a.geneSymbol, a.commonName, b.nodeID, b.commonName")
	gene_bp= pd.DataFrame(gene_bp)
	gene_bp_sub =gene_bp[gene_bp['a.geneSymbol'].isin(gene_list)]
	#print(gene_bp_sub)
	gene_bp_sub.to_csv(output_dir / f'{drug}_BP.csv', index=False)
	gene_bp_sub['b.commonName'].value_counts().to_csv(output_dir / f'{drug}_BP_valueCounts.csv', index=True)

	# find associated molecular function
	gene_mf =  alzkb.execute_and_fetch("MATCH (a:Gene)-[]-(b:MolecularFunction) Return a.nodeID,a.geneSymbol, a.commonName, b.nodeID, b.commonName, b.xrefGeneOntology")
	gene_mf= pd.DataFrame(gene_mf)
	gene_mf_sub =gene_mf[gene_mf['a.geneSymbol'].isin(gene_list)]
	#print(gene_mf_sub)
	gene_mf_sub.to_csv(output_dir / f'{drug}_MolecularFunction.csv', index=False)
	gene_mf_sub['b.commonName'].value_counts().to_csv(output_dir / f'{drug}_MolecularFunction_valueCounts.csv', index=True)

	# find associated TF
	gene_tf = alzkb.execute_and_fetch("MATCH (a:Gene)-[]-(b:TranscriptionFactor) Return a.nodeID,a.geneSymbol, a.commonName, b.nodeID, b.sourceDatabase, b.TF")
	gene_tf= pd.DataFrame(gene_tf)
	gene_tf_sub =gene_tf[gene_tf['a.geneSymbol'].isin(gene_list)]
	#print(gene_tf_sub)
	gene_tf_sub.to_csv(output_dir / f'{drug}_TranscriptionFactor.csv', index=False)
	gene_tf_sub['b.TF'].value_counts().to_csv(output_dir / f'{drug}_TranscriptionFactor_valueCounts.csv', index=True)

In [4]:
tpot_file_list = ['/Users/lib/Downloads/drug_analysis/exp1/sub_median.csv',
				  '/Users/lib/Downloads/drug_analysis/exp2/sub_median.csv',
				  '/Users/lib/Downloads/drug_analysis/exp3/sub_median.csv',
				  '/Users/lib/Downloads/drug_analysis/exp4/sub_median.csv']

for tpot_file in tpot_file_list:
	tpot_file = Path(tpot_file)
	tpot_file_dir = tpot_file.parent

	tmp = pd.read_csv(tpot_file)
	tmp['genes'] = tmp['genes'].apply(lambda x: x[1:-1].replace("'", "").split(', '))
	for row in tmp.itertuples():
		output_dir = tpot_file_dir / row.Name
		if output_dir.is_dir():
			print(f'Drug {row.Name} has been processed. Skipping.')
			continue

		print(f'Retrieving the AlzKB entities associated with the drug {row.Name}')
		output_dir.mkdir(parents=True)
		get_associated_entities(row.Name, row.genes, output_dir)


Retrieving the AlzKB entities associated with the drug DB00363
     a.nodeID a.xrefDrugbank a.commonName
5592   4485.0        DB00363    Clozapine
Retrieving the AlzKB entities associated with the drug DB00257
     a.nodeID a.xrefDrugbank  a.commonName
5586   5302.0        DB00257  Clotrimazole
Retrieving the AlzKB entities associated with the drug DB00368
      a.nodeID a.xrefDrugbank    a.commonName
11649   4470.0        DB00368  Norepinephrine
Retrieving the AlzKB entities associated with the drug DB00793
     a.nodeID a.xrefDrugbank a.commonName
8197   6212.0        DB00793   Haloprogin
Drug DB00793 has been processed. Skipping.
Retrieving the AlzKB entities associated with the drug DB00988
     a.nodeID a.xrefDrugbank a.commonName
6566   5584.0        DB00988     Dopamine
Drug DB00988 has been processed. Skipping.
Drug DB00988 has been processed. Skipping.
Retrieving the AlzKB entities associated with the drug DB00336
      a.nodeID a.xrefDrugbank a.commonName
11610   4634.0      

In [2]:
import pandas as pd
import math

results_dir = "/Users/lib/Library/CloudStorage/OneDrive-Cedars-SinaiHealthSystem/Orlenko, Alena's files - Network score/results/"
walk_methods = ["N2V", "E2V"]
feature_sets = ["Drug_analysis", "Pathway_analysis"]
experiments = ["exp1", "exp2", "exp3", "exp4"]

columns = ["Walk_method", "Feature_set", "Experiment", "Number_of_Pareto_front_models", "Unique_feature_sets", "Set_size_range", "Complexity_log_median", "Complexity_log_range", "Network_score_median", "Network_score_range", "Test_AUROC_median", "Test_AUROC_range"]
rez = pd.DataFrame(columns=columns)

count = 0
for walk_method in walk_methods:
    for feature_set in feature_sets:
        for experiment in experiments:
            file_path = f"{results_dir}/{walk_method}/{feature_set}/{experiment}/pareto_front_ext3.csv"
            df = pd.read_csv(file_path)
            rez.loc[count, "Walk_method"] = walk_method
            rez.loc[count, "Feature_set"] = feature_set
            rez.loc[count, "Experiment"] = experiment
            rez.loc[count, "Number_of_Pareto_front_models"] = len(df.Name)
            rez.loc[count, "Unique_feature_sets"] = df.Name.nunique()
            rez.loc[count, "Set_size_range"] = f'[{df["size"].min()}, {df["size"].max()}]'
            rez.loc[count, "Complexity_log_median"] = f'{round(math.log(df.Complexity.median()), 1)}'
            rez.loc[count, "Complexity_log_range"] = f'[{round(math.log(df.Complexity.min()), 1)}, {round(math.log(df.Complexity.max()),1)}]'
            rez.loc[count, "Network_score_median"] = f'{round(df.GS.median(), 1)}'
            rez.loc[count, "Network_score_range"] = f'[{round(df.GS.min(), 1)}, {round(df.GS.max(), 1)}]'
            rez.loc[count, "Test_AUROC_median"] = f'{round(df.test_score.median() * 100, 1)}'
            rez.loc[count, "Test_AUROC_range"] = f'[{round(df.test_score.min() * 100, 1)}, {round(df.test_score.max() * 100, 1)}]'
            count += 1

rez.to_csv(f"{results_dir}/summary_pareto_front_ext3.csv", index=False)

In [3]:
rez

,Walk_method,Feature_set,Experiment,Number_of_Pareto_front_models,Unique_feature_sets,Set_size_range,Complexity_log_median,Complexity_log_range,Network_score_median,Network_score_range,Test_AUROC_median,Test_AUROC_range
0,N2V,Drug_analysis,exp1,98,46,"[1, 174]",2.4,"[1.1, 13.6]",-0.4,"[-1.7, 3.0]",60.5,"[50.9, 63.9]"
1,N2V,Drug_analysis,exp2,97,54,"[1, 217]",2.2,"[1.1, 14.2]",0.1,"[0.1, 0.4]",60.1,"[50.8, 64.9]"
2,N2V,Drug_analysis,exp3,96,52,"[1, 217]",2.2,"[1.1, 14.2]",0.0,"[-3.0, 3.0]",59.9,"[50.5, 63.8]"
3,N2V,Drug_analysis,exp4,95,43,"[1, 180]",2.1,"[1.1, 13.9]",0.2,"[0.1, 0.4]",59.9,"[48.7, 64.2]"
4,N2V,Pathway_analysis,exp1,35,15,"[1, 21]",1.6,"[1.1, 13.3]",-0.9,"[-1.6, 2.0]",61.4,"[50.7, 62.6]"
5,N2V,Pathway_analysis,exp2,36,26,"[1, 31]",1.9,"[1.1, 13.2]",0.1,"[0.1, 0.4]",60.0,"[50.6, 62.0]"
6,N2V,Pathway_analysis,exp3,43,25,"[1, 28]",1.6,"[1.1, 13.4]",0.0,"[-1.3, 3.0]",59.2,"[50.6, 61.8]"
7,N2V,Pathway_analysis,exp4,28,14,"[1, 14]",1.7,"[1.1, 8.4]",0.2,"[0.1, 0.4]",59.6,"[50.9, 62.4]"
8,E2V,Drug_analysis,exp1,68,26,"[1, 172]",2.7,"[1.1, 13.6]",-0.3,"[-2.5, 3.0]",60.8,"[49.9, 64.0]"
9,E2V,Drug_analysis,exp2,56,39,"[1, 198]",2.4,"[1.1, 13.6]",0.6,"[0.4, 0.7]",59.9,"[49.9, 62.8]"
